# STEP 114A — Original Source Data Audit

**Project:** Logistics, Delivery & Returns Intelligence  
**Purpose:** Capture a reproducible baseline of all seven original datasets *before* final cleaning and validation.

This audit is **read-only**. It does not change the original CSVs, the Streamlit dashboard, or the client-facing PDF.

**What you will see:** (1) an inventory of source rows/columns, exact duplicate rows, and missing cells; (2) missing values by column. These are input-level observations, not proof that all cleaning tasks have been completed.

**Working rule:** An *exact duplicate* means every column of two rows contains equal values. We will separately validate business keys later.


In [1]:
# ============================================================
# STEP 114A — CELL 01: LOCATE AND READ THE SEVEN ORIGINAL CSVs
# ============================================================
# WHAT: Read the original, unmodified CSV files from Data/Raw.
# WHY: We need a trustworthy before-cleaning baseline for our log.
# SAFETY: This cell only READS files; nothing is overwritten or saved.

from pathlib import Path
import pandas as pd
from IPython.display import display

# Source names correspond to the original seven capstone datasets.
SOURCE_NAMES = (
    "customers", "products", "orders", "order_items",
    "payments", "shipments", "returns"
)

# Search the current folder and its parents to support a fresh GitHub clone.
# Retain the original iMac project path as a backward-compatible fallback.
working_folder = Path.cwd().resolve()
project_candidates = [
    working_folder,
    *working_folder.parents,
    Path.home() / "Documents" / "Python-Learning" / "Python-Capstone-Option-B"
]
PROJECT = next(
    (folder for folder in project_candidates
     if all((folder / "Data" / "Raw" / f"{name}.csv").is_file()
            for name in SOURCE_NAMES)),
    None
)
if PROJECT is None:
    raise FileNotFoundError(
        "Could not locate the seven Data/Raw source CSVs. "
        "Open Jupyter from the repository root or Notebook/ subfolder."
    )
RAW = PROJECT / "Data" / "Raw"

# Stop clearly if a source file is missing instead of silently skipping it.
missing_files = [f"{name}.csv" for name in SOURCE_NAMES if not (RAW / f"{name}.csv").is_file()]
print("Source directory:", RAW)
if missing_files:
    raise FileNotFoundError(
        "Missing original CSVs in Data/Raw: " + ", ".join(missing_files)
    )

# Keep the raw DataFrames separate from future cleaned working copies.
raw_tables = {
    name: pd.read_csv(RAW / f"{name}.csv", low_memory=False)
    for name in SOURCE_NAMES
}
print(f"Successfully opened {len(raw_tables)} original CSV files.")

Source directory: /Users/imacm3/Documents/Python-Learning/Python-Capstone-Option-B/Data/Raw
Successfully opened 7 original CSV files.


In [2]:
# ============================================================
# STEP 114A — CELL 02: DOCUMENT THE BASELINE DATA INVENTORY
# ============================================================
# WHAT: Count rows, columns, exact duplicate rows and blank/missing cells.
# WHY: These numbers will become the BEFORE column of our cleaning log.
# NOTE: A missing-cell count counts individual cells, not distinct rows.

inventory_rows = []
for name, df in raw_tables.items():
    inventory_rows.append({
        "Dataset": f"{name}.csv",
        "Raw rows": len(df),
        "Columns": len(df.columns),
        "Exact duplicate rows": int(df.duplicated().sum()),
        "Missing cells": int(df.isna().sum().sum()),
    })

source_inventory = pd.DataFrame(inventory_rows)
display(source_inventory)

print("Total original rows:", f"{source_inventory['Raw rows'].sum():,}")
print("Total exact duplicate rows:", f"{source_inventory['Exact duplicate rows'].sum():,}")
print("Reminder: This is an audit, not the final cleaned dataset.")

,Dataset,Raw rows,Columns,Exact duplicate rows,Missing cells
0,customers.csv,5050,13,50,971
1,products.csv,800,9,0,19
2,orders.csv,15075,8,75,1232
3,order_items.csv,37763,5,262,2262
4,payments.csv,15000,6,0,878
5,shipments.csv,15000,7,0,729
6,returns.csv,1200,6,0,297


Total original rows: 89,888
Total exact duplicate rows: 387
Reminder: This is an audit, not the final cleaned dataset.


In [3]:
# ============================================================
# STEP 114A — CELL 03: IDENTIFY WHERE VALUES ARE MISSING
# ============================================================
# WHAT: List every column containing one or more missing values.
# WHY: Next, we will choose and document a field-specific fill/drop/
#      retain-and-flag rule instead of deleting records blindly.
# SAFETY: This cell ONLY REPORTS missing data; it does not fill/drop it.

missing_rows = []
for name, df in raw_tables.items():
    counts = df.isna().sum()
    for column, count in counts.items():
        if count > 0:
            missing_rows.append({
                "Dataset": f"{name}.csv",
                "Column": column,
                "Missing values": int(count),
                "Missing % of rows": round(count / len(df) * 100, 2),
            })

missing_value_inventory = pd.DataFrame(missing_rows)
missing_value_inventory = missing_value_inventory.sort_values(
    ["Dataset", "Missing values"], ascending=[True, False]
).reset_index(drop=True)
display(missing_value_inventory)
print(f"Columns with missing values: {len(missing_value_inventory)}")
print("NEXT: STEP 114B will assign and validate cleaning rules by field.")

,Dataset,Column,Missing values,Missing % of rows
0,customers.csv,gender,301,5.96
1,customers.csv,loyalty_tier,266,5.27
2,customers.csv,age,150,2.97
3,customers.csv,phone,147,2.91
4,customers.csv,email,107,2.12
5,order_items.csv,unit_price_at_purchase,2262,5.99
6,orders.csv,discount_pct,758,5.03
7,orders.csv,shipping_fee,474,3.14
8,payments.csv,amount_paid,878,5.85
9,products.csv,stock_qty,19,2.38


Columns with missing values: 13
NEXT: STEP 114B will assign and validate cleaning rules by field.


## What to send back

Run **Run → Run All Cells**. Send a screenshot showing the **seven-row source inventory** (Cell 02). If an error appears instead, send the full error message.

Do **not** delete, replace, or overwrite the original `Data/Raw/*.csv` files. Step 114B will use this audit to decide exact cleaning actions and record their before/after evidence.
